# 01 - TF-IDF baseline on cleaned full AllNLI

Notebook nay chay doc lap tren Kaggle. TF-IDF la baseline khong lap theo epoch nen khong co early stopping; notebook van danh gia test 5k va thoi gian scoring tren test 5k.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/similarity_search')
GITHUB_REPOSITORY_URL = 'https://github.com/PhDQuang/similarity_search.git'

if not PROJECT_ROOT.exists():
    !git clone {GITHUB_REPOSITORY_URL} {PROJECT_ROOT}

%cd {PROJECT_ROOT}
%pip install -q -r fix/requirements-kaggle.txt
%pip install -q -e fix

Cloning into '/kaggle/working/similarity_search'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 222 (delta 90), reused 185 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 481.65 KiB | 18.52 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/kaggle/working/similarity_search
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for similarity-search-fix (pyproject.toml) ... done
Note: y

In [2]:
from pathlib import Path
import shutil

CLEAN_DATA_DIR = Path('fix/data/processed/allnli_70_15_15_clean/pair-class')
KAGGLE_CLEAN_CANDIDATES = [
    Path('/kaggle/input/allnli-70-15-15-clean/pair-class'),
    Path('/kaggle/input/allnli-70-15-15-clean/allnli_70_15_15_clean/pair-class'),
]

def has_clean_data(path: Path) -> bool:
    return all((path / f'{split}.parquet').exists() for split in ('train', 'val', 'test'))

if not has_clean_data(CLEAN_DATA_DIR):
    source = next((path for path in KAGGLE_CLEAN_CANDIDATES if has_clean_data(path)), None)
    if source is not None:
        CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
        for item in source.iterdir():
            if item.is_file():
                shutil.copy2(item, CLEAN_DATA_DIR / item.name)
    else:
        !python -m similarity_search_fix.data.prepare_allnli_70_15_15_clean --output-dir {CLEAN_DATA_DIR} --seed 42

assert has_clean_data(CLEAN_DATA_DIR), f'Missing clean data: {CLEAN_DATA_DIR}'
print('Using clean data:', CLEAN_DATA_DIR)

README.md: 5.15kB [00:00, 21.9MB/s]
pair-class/train-00000-of-00001.parquet: 100%|█| 69.5M/69.5M [00:03<00:00, 22.8M
pair-class/dev-00000-of-00001.parquet: 100%|█| 1.57M/1.57M [00:00<00:00, 3.81MB/
pair-class/test-00000-of-00001.parquet: 100%|█| 1.61M/1.61M [00:00<00:00, 2.62MB
Generating train split: 100%|█| 942069/942069 [00:00<00:00, 1349996.54 examples/
Generating test split: 100%|██| 19656/19656 [00:00<00:00, 1116572.40 examples/s]
{
  "dataset_name": "sentence-transformers/all-nli",
  "dataset_config": "pair-class",
  "created_at_utc": "2026-07-05T14:47:00.959752+00:00",
  "seed": 42,
  "split_ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "saved_paths": {
    "train": "fix/data/processed/allnli_70_15_15_clean/pair-class/train.parquet",
    "val": "fix/data/processed/allnli_70_15_15_clean/pair-class/val.parquet",
    "test": "fix/data/processed/allnli_70_15_15_clean/pair-class/test.parquet"
  },
  "row_counts": {
    "train": 686442,
    "val": 147033,
    

In [3]:
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/fix_outputs/tfidf_clean_full')
MODEL_DIR = Path('/kaggle/working/fix_models/tfidf_clean_full')

!python -m similarity_search_fix.models.train_tfidf \
  --input-dir {CLEAN_DATA_DIR} \
  --output-dir {OUTPUT_DIR} \
  --model-dir {MODEL_DIR} \
  --max-features 50000 \
  --ngram-max 2 \
  --max-retrieval-queries 1000 \
  --test-sample-size 5000 \
  --seed 42

{'task': 'entailment-as-semantic-similarity', 'fixed_dataset': 'AllNLI pair-class full 70/15/15', 'positive_label': 'entailment', 'model': {'name': 'TF-IDF baseline', 'trained_in_project': True, 'vocabulary_size': 50000, 'ngram_range': [1, 2], 'max_features': 50000, 'min_df': 2, 'lowercase_done_in_dataset': True, 'stop_words': 'english'}, 'threshold_selection': {'split': 'val', 'threshold': 0.08600753671797215, 'best_f1': 0.5211636363636363}, 'pair_classification': {'val': {'threshold': 0.08600753671797215, 'accuracy': 0.5074303047615161, 'precision': 0.38530648157200115, 'recall': 0.8050040849673202, 'f1': 0.5211636363636364, 'average_precision': 0.4634825820739, 'mean_positive_score': 0.35945701567214994, 'mean_negative_score': 0.23587363856198587, 'roc_auc': 0.6365559073135064}, 'test': {'threshold': 0.08600753671797215, 'accuracy': 0.5062702003878474, 'precision': 0.3850381709198118, 'recall': 0.804770552347528, 'f1': 0.5208692379311028, 'average_precision': 0.4579403676352435, 'me

In [4]:
from pathlib import Path
import json
import shutil

ARTIFACT_DIR = Path('/kaggle/working/artifacts_tfidf_clean_full')
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)
shutil.copytree(OUTPUT_DIR, ARTIFACT_DIR / 'outputs')
shutil.copytree(MODEL_DIR, ARTIFACT_DIR / 'model')
zip_path = shutil.make_archive(str(ARTIFACT_DIR), 'zip', ARTIFACT_DIR)
print('Download artifact:', zip_path)

display(json.loads((OUTPUT_DIR / 'metrics.json').read_text()))
display(json.loads((OUTPUT_DIR / 'test5k_performance.json').read_text()))

Download artifact: /kaggle/working/artifacts_tfidf_clean_full.zip


{'task': 'entailment-as-semantic-similarity',
 'fixed_dataset': 'AllNLI pair-class full 70/15/15',
 'positive_label': 'entailment',
 'model': {'name': 'TF-IDF baseline',
  'trained_in_project': True,
  'vocabulary_size': 50000,
  'ngram_range': [1, 2],
  'max_features': 50000,
  'min_df': 2,
  'lowercase_done_in_dataset': True,
  'stop_words': 'english'},
 'threshold_selection': {'split': 'val',
  'threshold': 0.08600753671797215,
  'best_f1': 0.5211636363636363},
 'pair_classification': {'val': {'threshold': 0.08600753671797215,
   'accuracy': 0.5074303047615161,
   'precision': 0.38530648157200115,
   'recall': 0.8050040849673202,
   'f1': 0.5211636363636364,
   'average_precision': 0.4634825820739,
   'mean_positive_score': 0.35945701567214994,
   'mean_negative_score': 0.23587363856198587,
   'roc_auc': 0.6365559073135064},
  'test': {'threshold': 0.08600753671797215,
   'accuracy': 0.5062702003878474,
   'precision': 0.3850381709198118,
   'recall': 0.804770552347528,
   'f1': 0.5

{'sample_rows': 5000,
 'elapsed_seconds': 0.13745563599991328,
 'pairs_per_second': 36375.37277848072,
 'threshold': 0.08600753671797215,
 'pair_classification': {'threshold': 0.08600753671797215,
  'accuracy': 0.4978,
  'precision': 0.38040676024061876,
  'recall': 0.7923627684964201,
  'f1': 0.5140313528159474,
  'average_precision': 0.45651558632293143,
  'mean_positive_score': 0.3571169065771811,
  'mean_negative_score': 0.24206410939197892,
  'roc_auc': 0.6251795002139643}}